# O Ônibus do Ponto 348: render no Colab (à prova de queda)

Gera o `mp4` de cada episódio (60 fps, 60 s, com a mixagem final) numa GPU do Colab, em **lotes curtos** com checkpoint no Google Drive.

**Primeira vez:** *Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)* e rode a célula **▶ RETOMAR**.
Ela pede para você escolher o `vozes-finais.zip` (botão *Salvar vozes* da página das prévias); depois disso o arquivo fica no Drive.

**Sempre que a sessão cair:** reconecte (de novo em GPU) e rode **só a célula ▶ RETOMAR**. Ela refaz a instalação (~1 min, o Blender fica em cache no Drive)
e continua **do primeiro quadro que falta**. Nada é refeito: cada quadro pronto já está no Drive.

Onde fica tudo (Google Drive › `ponto348`):
- `out/quadros/epXX/`: os quadros prontos (checkpoint);  `out/quadros/epXX.log`: um registro por lote, com s/quadro, RAM e VRAM
- `out/epXX.mp4`: o vídeo pronto (um episódio com mp4 é pulado nas próximas rodadas)
- `vozes-finais.zip` e `cache/`: vozes e Blender, para não repetir downloads

## ▶ RETOMAR (rode esta, sempre)

In [ ]:
# ===== AJUSTES (a T4 do Colab aguenta bem 720p; mude só se quiser) =====
EPS     = "ep01 ep02 ep03"   # em ordem; episódio com mp4 pronto é pulado
WIDTH   = "720"              # 720 = bom custo-benefício; 1080 = máxima (bem mais lento e mais espaço no Drive)
SAMPLES = "24"               # amostras por pixel: T4 em 720p = 16 a 32 (menos = mais rápido, mais ruído)
STEP    = "2"               # 2 = 30 fps (metade do trabalho; o TikTok aceita); 1 = 60 fps
BATCH   = "30"               # quadros por lote; se a sessão cair muito, use 15
# =======================================================================
import os, glob, shutil, zipfile, subprocess
from google.colab import drive, files

if subprocess.run(['nvidia-smi', '-L'], capture_output=True).returncode != 0:
    raise RuntimeError('Sem GPU: Ambiente de execução > Alterar tipo de ambiente de execução > GPU (T4). Depois rode esta célula de novo.')

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/ponto348'
OUT = f'{BASE}/out'
REPO = '/content/ponto348_repo'
BL = '/content/blender/blender'
os.makedirs(f'{OUT}/quadros', exist_ok=True)
os.makedirs(f'{BASE}/cache', exist_ok=True)

# projeto (repositório público): sempre a versão mais nova da branch
BRANCH = 'claude/loving-franklin-rmvp6i'
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} fetch -q --depth 1 origin {BRANCH} && git -C {REPO} reset -q --hard FETCH_HEAD
else:
    !git clone -q --branch {BRANCH} --depth 1 https://github.com/nunnsx-pixel/Projectonesuspense {REPO}
!git -C {REPO} log --oneline -1

# Blender 4.2 LTS (o .tar.xz fica no Drive: depois de uma queda só descompacta)
!bash {REPO}/blender/tools/instalar_blender.sh {BASE}/cache /content/blender

# vozes (fora do git): do Drive, ou escolha o zip na primeira vez
zip_drive = f'{BASE}/vozes-finais.zip'
if not os.path.exists(zip_drive):
    print('Primeira vez: escolha o arquivo vozes-finais.zip')
    os.chdir('/content')
    up = files.upload()
    shutil.copy(next(iter(up)), zip_drive)
with zipfile.ZipFile(zip_drive) as z:
    z.extractall(REPO)
n_voz = len(glob.glob(f'{REPO}/ep0*/audio/*.mp3'))
assert n_voz == 11, f'faltam vozes: {n_voz}/11 (confira o vozes-finais.zip)'

# texturas CC0: guardadas no Drive (cache) e copiadas para o disco local, que é mais rápido de ler a cada lote
TEX, TEX_CACHE = '/content/textures', f'{BASE}/cache/textures'
if os.path.isdir(TEX_CACHE) and not os.path.isdir(TEX):
    shutil.copytree(TEX_CACHE, TEX)
!python3 {REPO}/blender/tools/fetch_textures.py --dest {TEX}
if not os.path.isdir(TEX_CACHE):
    shutil.copytree(TEX, TEX_CACHE)

# render em lotes: o Blender abre e fecha a cada BATCH quadros; o perfil t4 poupa RAM/VRAM; os quadros vão para o Drive a cada lote
!cd {REPO} && EPS="{EPS}" WIDTH={WIDTH} SAMPLES={SAMPLES} BATCH={BATCH} STEP={STEP} PERFIL=t4 STAGE_DIR=/content/stage TEXTURES_DIR={TEX} BLENDER={BL} GPU_BACKEND=cuda RENDERS_DIR={OUT}/quadros OUT_DIR={OUT} ./render_tudo.sh
print('\nPRONTO por esta rodada. Vídeos em', OUT)
!ls -la {OUT}/*.mp4

## Ver o andamento (rápido, a qualquer momento)

Mostra quantos quadros já estão no Drive e os últimos registros (s/quadro, RAM e VRAM). Se a sessão acabou de cair, rode antes o mount do Drive abaixo (já incluso).

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ponto348/out'
for ep in ('ep01', 'ep02', 'ep03'):
    pasta = f'{OUT}/quadros/{ep}'
    n = len([f for f in os.listdir(pasta) if f.endswith('.png')]) if os.path.isdir(pasta) else 0
    total = 3600 // 2 if globals().get('STEP', '2') == '2' else 3600
    pronto = os.path.exists(f'{OUT}/{ep}.mp4')
    print(f'{ep}: ' + ('mp4 PRONTO' if pronto else f'{n}/{total} quadros ({100 * n // total}%)'))
    log = f'{OUT}/quadros/{ep}.log'
    if os.path.exists(log):
        print('   ' + open(log, encoding='utf-8').read().strip().splitlines()[-1])

## (Opcional) Medir a velocidade

Rode **depois** da célula ▶ RETOMAR ter instalado tudo (mesma sessão). Renderiza 6 quadros do Ep. 1 e projeta as horas do episódio inteiro.
Use para decidir `WIDTH`/`SAMPLES`.

In [ ]:
!cd {REPO} && BLENDER={BL} GPU_BACKEND=cuda PERFIL=t4 TEXTURES_DIR={TEX} STEP={STEP} BENCH=1 EPS=ep01 WIDTH={WIDTH} SAMPLES={SAMPLES} RENDERS_DIR=/content/bench ./render_tudo.sh

## (Opcional) Liberar espaço no Drive

Apaga os quadros dos episódios que **já têm mp4 pronto**. No Drive isso vai para a lixeira: esvazie-a depois para recuperar o espaço (plano grátis = 15 GB).

In [ ]:
import os, shutil
OUT = '/content/drive/MyDrive/ponto348/out'
for ep in ('ep01', 'ep02', 'ep03'):
    pasta = f'{OUT}/quadros/{ep}'
    if os.path.exists(f'{OUT}/{ep}.mp4') and os.path.isdir(pasta):
        shutil.rmtree(pasta)
        print('apagados os quadros de', ep)